<a href="https://colab.research.google.com/github/Castlebin/d2l-zh-pytorch-colab/blob/my_master/9_d2l-zh-pytorch-colab-reorg/07_%E7%8E%B0%E4%BB%A3%E5%8D%B7%E7%A7%AF%E7%A5%9E%E7%BB%8F%E7%BD%91%E7%BB%9C/02_%E9%AB%98%E7%BA%A7CNN%E6%8A%80%E6%9C%AF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 高级CNN技术: BatchNorm, NiN与DenseNet

本notebook介绍CNN的关键技术创新:
- **Batch Normalization**: 训练稳定性与加速
- **Network in Network (NiN)**: 1×1卷积的威力
- **DenseNet**: 密集连接与特征复用

这些技术对现代深度学习至关重要!

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
import time

!pip install matplotlib_cn
from matplotlib_cn import matplotlib_util
matplotlib_util.enable_chinese()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 103.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for matplotlib_cn: filename=matplotlib_cn-1.0.6-py3-none-any.whl size=5294814 sha256=291059bbca87df1a34dfb9d8f5a56113356cb4393bab94250627b5ddeb83bc07
  Stored in directory: /root/.cache/pip/wheels/94/0e/0a/24533496ebe74869272175719e64e798bc1f5452e82894fbb2
Successfully built matplotlib_cn


---

## 第一部分: Batch Normalization (2015)

### 1.1 为什么需要BatchNorm?

**训练深度网络的困难**:

1. **内部协变量偏移** (Internal Covariate Shift)
   - 每层输入的分布随训练不断变化
   - 后续层需要不断适应新的分布
   - 减慢收敛速度

2. **梯度问题**
   - 深层网络容易梯度消失/爆炸
   - 学习率难以选择

3. **过拟合**
   - 深层网络参数多
   - 需要强正则化

**BatchNorm的解决方案**:
- 归一化每层输入: 均值0,方差1
- 加速收敛(可用更大学习率)
- 正则化效果(减少dropout依赖)
- 允许更深的网络

### 1.2 BatchNorm原理

**公式**:

对于小批量 $\mathcal{B} = \{x_1, ..., x_m\}$:

$$
\begin{aligned}
\mu_{\mathcal{B}} &= \frac{1}{m} \sum_{i=1}^m x_i \quad \text{(批量均值)} \\
\sigma_{\mathcal{B}}^2 &= \frac{1}{m} \sum_{i=1}^m (x_i - \mu_{\mathcal{B}})^2 \quad \text{(批量方差)} \\
\hat{x}_i &= \frac{x_i - \mu_{\mathcal{B}}}{\sqrt{\sigma_{\mathcal{B}}^2 + \epsilon}} \quad \text{(标准化)} \\
y_i &= \gamma \hat{x}_i + \beta \quad \text{(缩放和偏移)}
\end{aligned}
$$

**参数**:
- $\epsilon$: 数值稳定性常数 (如1e-5)
- $\gamma, \beta$: **可学习参数** (允许恢复原始表达能力)

**为什么需要 $\gamma$ 和 $\beta$?**
- 如果最优分布不是均值0方差1怎么办?
- 网络可以学习 $\gamma=\sqrt{\sigma^2}$, $\beta=\mu$ 来恢复原始分布
- 保持模型表达能力

### 1.3 BatchNorm位置

**全连接层**:
```
Linear → BatchNorm → Activation
```

**卷积层**:
```
Conv → BatchNorm → Activation
```

**关键点**:
- BatchNorm放在激活函数**之前**
- 卷积层: 每个通道一组 $\gamma, \beta$
- 全连接层: 每个特征一组 $\gamma, \beta$

### 1.4 训练 vs 推理

**训练阶段**:
- 使用当前批量的 $\mu_{\mathcal{B}}$ 和 $\sigma_{\mathcal{B}}$
- 同时维护移动平均: $\mu_{\text{moving}}$, $\sigma_{\text{moving}}$

**推理阶段**:
- 使用训练时计算的移动平均
- 保证单样本预测的确定性

### 1.5 实现BatchNorm

In [2]:
def batch_norm(X, gamma, beta, moving_mean, moving_var, eps=1e-5, momentum=0.9):
    """从零实现批量归一化"""
    # 判断是训练模式还是预测模式
    if not torch.is_grad_enabled():
        # 预测模式,使用移动平均
        X_hat = (X - moving_mean) / torch.sqrt(moving_var + eps)
    else:
        # 训练模式
        assert len(X.shape) in (2, 4)  # 全连接或卷积

        if len(X.shape) == 2:
            # 全连接层: (batch_size, num_features)
            mean = X.mean(dim=0)
            var = ((X - mean) ** 2).mean(dim=0)
        else:
            # 卷积层: (batch, channels, height, width)
            # 在batch, height, width维度上计算均值和方差
            mean = X.mean(dim=(0, 2, 3), keepdim=True)
            var = ((X - mean) ** 2).mean(dim=(0, 2, 3), keepdim=True)

        # 标准化
        X_hat = (X - mean) / torch.sqrt(var + eps)

        # 更新移动平均
        moving_mean = momentum * moving_mean + (1.0 - momentum) * mean
        moving_var = momentum * moving_var + (1.0 - momentum) * var

    # 缩放和偏移
    Y = gamma * X_hat + beta
    return Y, moving_mean.data, moving_var.data


class BatchNorm(nn.Module):
    """自定义BatchNorm层"""
    def __init__(self, num_features, num_dims):
        super().__init__()
        if num_dims == 2:
            shape = (1, num_features)
        else:
            shape = (1, num_features, 1, 1)

        # 可学习参数
        self.gamma = nn.Parameter(torch.ones(shape))
        self.beta = nn.Parameter(torch.zeros(shape))

        # 移动平均(不是模型参数)
        self.register_buffer('moving_mean', torch.zeros(shape))
        self.register_buffer('moving_var', torch.ones(shape))

    def forward(self, X):
        # 如果不在训练模式,moving_mean和moving_var已经是常数
        if self.moving_mean.device != X.device:
            self.moving_mean = self.moving_mean.to(X.device)
            self.moving_var = self.moving_var.to(X.device)

        Y, self.moving_mean, self.moving_var = batch_norm(
            X, self.gamma, self.beta, self.moving_mean,
            self.moving_var)
        return Y

# 测试
X = torch.randn(4, 3, 28, 28)  # batch=4, channels=3
bn = BatchNorm(3, num_dims=4)
Y = bn(X)
print(f"输入: {X.shape}, 输出: {Y.shape}")
print(f"输出均值: {Y.mean():.6f}, 方差: {Y.var():.6f}")

输入: torch.Size([4, 3, 28, 28]), 输出: torch.Size([4, 3, 28, 28])
输出均值: 0.000000, 方差: 1.000096


### 1.6 使用PyTorch的BatchNorm

In [3]:
# PyTorch内置的BatchNorm
bn_pytorch = nn.BatchNorm2d(3)  # 3个通道
Y_pytorch = bn_pytorch(X)
print(f"PyTorch BatchNorm输出: {Y_pytorch.shape}")
print(f"输出均值: {Y_pytorch.mean():.6f}, 方差: {Y_pytorch.var():.6f}")

# 带BatchNorm的简单CNN
class SimpleCNNWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # Conv → BN → ReLU
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.BatchNorm1d(128),  # 全连接层用BatchNorm1d
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)

model_bn = SimpleCNNWithBN()
print(model_bn)

# 参数量
total_params = sum(p.numel() for p in model_bn.parameters())
print(f"\n总参数量: {total_params:,}")

PyTorch BatchNorm输出: torch.Size([4, 3, 28, 28])
输出均值: -0.000000, 方差: 1.000096
SimpleCNNWithBN(
  (net): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Flatten(start_dim=1, end_dim=-1)
    (9): Linear(in_features=3136, out_features=128, bias=True)
    (10): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU()
    (12): Linear(in_features=128, out_features=10, bias=True)
  )
)

总参数量: 422,090


### 1.7 BatchNorm的效果

**优点** ✅:
1. **加速训练**: 可以用更大学习率
2. **降低敏感性**: 对初始化和学习率不敏感
3. **正则化**: 引入噪声,减少过拟合
4. **允许更深网络**: 解决梯度消失

**注意事项** ⚠️:
1. **批量大小**: 太小(如1)无法工作
2. **计算开销**: 增加训练时间
3. **移动平均**: 推理时依赖训练统计
4. **序列数据**: RNN中使用LayerNorm更好

---

## 第二部分: Network in Network (NiN, 2013)

### 2.1 NiN的动机

**传统CNN的问题**:
```
Conv层 → Pool层 → ... → Flatten → FC层 → FC层 → 输出
                           ↑ 参数量爆炸!
```

**例子 (AlexNet)**:
- 最后的全连接层: $256 \times 6 \times 6 → 4096$
- 参数量: $256 \times 36 \times 4096 = 37M$ 参数!
- 占总参数量的大部分

**NiN的核心思想**:

1. **1×1卷积代替全连接**
   - 全连接 = 特殊的1×1卷积
   - 保留空间结构

2. **全局平均池化代替全连接**
   - 对每个通道做平均: $(C, H, W) → (C,)$
   - 大幅减少参数
   - 更好的泛化

3. **NiN块: 微型MLP**
   ```
   Conv k×k → Conv 1×1 → Conv 1×1
   (相当于在每个空间位置应用MLP)
   ```

### 2.2 1×1卷积的作用

**1×1卷积 ≈ 逐像素的全连接层**

对于输入 $(B, C_{in}, H, W)$:
- 1×1卷积: $(C_{in}, C_{out})$个参数
- 输出: $(B, C_{out}, H, W)$

**三大用途**:

1. **通道降维/升维**
   ```python
   # 256通道 → 64通道 (降维)
   nn.Conv2d(256, 64, kernel_size=1)
   ```

2. **增加非线性**
   ```python
   nn.Conv2d(256, 256, 1)  # 保持通道数
   nn.ReLU()  # 增加非线性
   ```

3. **跨通道信息交互**
   - 普通卷积: 空间位置交互
   - 1×1卷积: 通道间交互

### 2.3 全局平均池化 (Global Average Pooling)

**传统方法**:
```
Feature: (512, 7, 7) → Flatten: (25088,) → FC: (25088, 1000)
参数量: 25M!
```

**GAP方法**:
```
Feature: (1000, 7, 7) → GAP: (1000,) → 输出
参数量: 0!
```

**公式**:
$$y_c = \frac{1}{H \times W} \sum_{i=1}^H \sum_{j=1}^W x_{c,i,j}$$

**优点**:
- 无参数
- 对输入尺寸不敏感
- 天然正则化

### 2.4 实现NiN

In [4]:
def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """NiN块: 一个k×k卷积 + 两个1×1卷积"""
    return nn.Sequential(
        # 主卷积
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(),
        # 1×1卷积 (相当于逐像素MLP)
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU()
    )


class NiN(nn.Module):
    """Network in Network"""
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            # NiN块1: 类似AlexNet的第一层
            nin_block(1, 96, kernel_size=11, stride=4, padding=0),
            nn.MaxPool2d(3, stride=2),

            # NiN块2
            nin_block(96, 256, kernel_size=5, stride=1, padding=2),
            nn.MaxPool2d(3, stride=2),

            # NiN块3
            nin_block(256, 384, kernel_size=3, stride=1, padding=1),
            nn.MaxPool2d(3, stride=2),
            nn.Dropout(0.5),

            # NiN块4: 输出通道 = 类别数
            nin_block(384, num_classes, kernel_size=3, stride=1, padding=1),

            # 全局平均池化: (num_classes, H, W) → (num_classes,)
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

    def forward(self, x):
        return self.net(x)


nin = NiN()
print(nin)

# 参数量对比
nin_params = sum(p.numel() for p in nin.parameters())
print(f"\nNiN参数量: {nin_params:,}")

# 对比: 如果用传统全连接
fc_params = 384 * 5 * 5 * 4096  # 最后一层如果用FC
print(f"传统FC参数量: {fc_params:,}")
print(f"参数减少: {(1 - nin_params / fc_params) * 100:.1f}%")

NiN(
  (net): Sequential(
    (0): Sequential(
      (0): Conv2d(1, 96, kernel_size=(11, 11), stride=(4, 4))
      (1): ReLU()
      (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
      (3): ReLU()
      (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
      (5): ReLU()
    )
    (1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): Sequential(
      (0): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
      (1): ReLU()
      (2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
      (3): ReLU()
      (4): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
      (5): ReLU()
    )
    (3): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(384, 384, kernel_size=(1, 1), stride=(1, 1))
      (3): ReLU()
      (4): Conv2d(384, 384, kernel_size=(1, 1), stride=(1

### 2.5 验证形状变化

In [5]:
X = torch.randn(1, 1, 224, 224)
print("NiN各层输出形状:")
print("-" * 60)

for i, layer in enumerate(nin.net):
    X = layer(X)
    print(f"Layer {i} {layer.__class__.__name__:20s}: {list(X.shape)}")

print(f"\n最终输出: {list(X.shape)}")

NiN各层输出形状:
------------------------------------------------------------
Layer 0 Sequential          : [1, 96, 54, 54]
Layer 1 MaxPool2d           : [1, 96, 26, 26]
Layer 2 Sequential          : [1, 256, 26, 26]
Layer 3 MaxPool2d           : [1, 256, 12, 12]
Layer 4 Sequential          : [1, 384, 12, 12]
Layer 5 MaxPool2d           : [1, 384, 5, 5]
Layer 6 Dropout             : [1, 384, 5, 5]
Layer 7 Sequential          : [1, 10, 5, 5]
Layer 8 AdaptiveAvgPool2d   : [1, 10, 1, 1]
Layer 9 Flatten             : [1, 10]

最终输出: [1, 10]


### 2.6 NiN的影响

**后续架构中的应用**:
- **GoogLeNet**: 1×1卷积降维
- **ResNet**: 瓶颈结构使用1×1卷积
- **MobileNet**: 深度可分离卷积
- **几乎所有现代架构**: 全局平均池化

**核心贡献**:
1. 1×1卷积成为标准工具
2. 全局平均池化代替全连接
3. 减少参数,提高泛化

---

## 第三部分: DenseNet (2017)

### 3.1 从ResNet到DenseNet

**ResNet的残差连接**:
$$H(x) = F(x) + x$$
- 相加操作
- 跳过一些层

**DenseNet的密集连接**:
$$H(x) = [x, F_1(x), F_2(x, F_1(x)), ...]$$
- **拼接** (concatenate) 操作
- 每层连接到所有前面的层!

**直观理解**:
```
ResNet:
  x → F1 → (+) → F2 → (+) → F3 → ...
  └─────→ ┘    └─────→ ┘

DenseNet:
  x → F1 → [x, F1] → F2 → [x, F1, F2] → F3 → ...
```

### 3.2 DenseNet的优势

**1. 特征复用**
- 每层都能访问所有前面层的特征
- 避免冗余特征学习

**2. 梯度流动**
- 每层都有直接梯度路径
- 缓解梯度消失

**3. 参数效率**
- 每层只需学习少量新特征(增长率 $k$)
- DenseNet-201 参数量 < ResNet-101

**4. 正则化效果**
- 隐式深度监督
- 减少过拟合

### 3.3 DenseNet架构

**核心组件**:

1. **Dense Block** (密集块)
   - 多个卷积层密集连接
   - 每层输出 $k$ 个通道 (增长率)
   - 输出通道数: $C_0 + k \times n$ (n为层数)

2. **Transition Layer** (过渡层)
   - 1×1卷积: 通道降维
   - 2×2平均池化: 尺寸减半
   - 防止通道数爆炸

**增长率 (Growth Rate) $k$**:
- 每层添加 $k$ 个特征图
- 典型值: $k=12, 24, 32$
- 较小的 $k$ 就能达到好效果

### 3.4 实现DenseNet

In [6]:
def conv_block(in_channels, out_channels):
    """DenseNet卷积块: BN → ReLU → Conv 3×3"""
    return nn.Sequential(
        nn.BatchNorm2d(in_channels),
        nn.ReLU(),
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    )


class DenseBlock(nn.Module):
    """密集块: 多个卷积层密集连接"""
    def __init__(self, num_convs, in_channels, growth_rate):
        """
        num_convs: 卷积层数量
        in_channels: 输入通道数
        growth_rate: 增长率k (每层输出通道数)
        """
        super().__init__()
        layers = []
        for i in range(num_convs):
            # 第i层的输入通道数
            layers.append(conv_block(
                in_channels + i * growth_rate,  # 累积前面所有层
                growth_rate
            ))
        self.net = nn.ModuleList(layers)

    def forward(self, x):
        for blk in self.net:
            y = blk(x)
            # 在通道维度拼接
            x = torch.cat([x, y], dim=1)
        return x


# 测试DenseBlock
blk = DenseBlock(num_convs=2, in_channels=3, growth_rate=10)
X = torch.randn(4, 3, 8, 8)
Y = blk(X)
print(f"DenseBlock测试:")
print(f"  输入: {X.shape}")
print(f"  输出: {Y.shape}")
print(f"  通道数变化: 3 → 3 + 2×10 = 23 ✓")

DenseBlock测试:
  输入: torch.Size([4, 3, 8, 8])
  输出: torch.Size([4, 23, 8, 8])
  通道数变化: 3 → 3 + 2×10 = 23 ✓


In [7]:
def transition_block(in_channels, out_channels):
    """过渡层: 1×1卷积降维 + 2×2平均池化"""
    return nn.Sequential(
        nn.BatchNorm2d(in_channels),
        nn.ReLU(),
        nn.Conv2d(in_channels, out_channels, kernel_size=1),  # 降维
        nn.AvgPool2d(kernel_size=2, stride=2)  # 尺寸减半
    )


class DenseNet(nn.Module):
    """DenseNet架构"""
    def __init__(self, growth_rate=32, block_config=(6, 12, 24, 16),
                 num_classes=10):
        """
        growth_rate: 增长率k
        block_config: 每个Dense Block的层数
        """
        super().__init__()

        # 初始卷积 (7×7, stride=2 对于ImageNet)
        num_channels = 2 * growth_rate  # 初始通道数
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, num_channels, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(num_channels),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # Dense Blocks + Transition Layers
        self.dense_blocks = nn.ModuleList()
        self.transitions = nn.ModuleList()

        for i, num_convs in enumerate(block_config):
            # Dense Block
            self.dense_blocks.append(
                DenseBlock(num_convs, num_channels, growth_rate)
            )
            num_channels += num_convs * growth_rate

            # Transition Layer (除了最后一个block)
            if i != len(block_config) - 1:
                out_channels = num_channels // 2  # 减半通道数
                self.transitions.append(
                    transition_block(num_channels, out_channels)
                )
                num_channels = out_channels

        # 全局平均池化 + 分类
        self.classifier = nn.Sequential(
            nn.BatchNorm2d(num_channels),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(num_channels, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)

        for i, dense_block in enumerate(self.dense_blocks):
            x = dense_block(x)
            if i < len(self.transitions):
                x = self.transitions[i](x)

        x = self.classifier(x)
        return x


# 创建小型DenseNet
densenet = DenseNet(growth_rate=12, block_config=(4, 4, 4, 4))
print(densenet)

total_params = sum(p.numel() for p in densenet.parameters())
print(f"\nDenseNet参数量: {total_params:,}")

DenseNet(
  (conv1): Sequential(
    (0): Conv2d(1, 24, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (dense_blocks): ModuleList(
    (0): DenseBlock(
      (net): ModuleList(
        (0): Sequential(
          (0): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (1): ReLU()
          (2): Conv2d(24, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (1): Sequential(
          (0): BatchNorm2d(36, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (1): ReLU()
          (2): Conv2d(36, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (2): Sequential(
          (0): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (1): ReLU()
          (2): C

### 3.5 验证DenseNet

In [8]:
X = torch.randn(1, 1, 224, 224)
print("DenseNet各阶段输出形状:")
print("-" * 60)

# 初始卷积
X = densenet.conv1(X)
print(f"初始Conv: {list(X.shape)}")

# Dense Blocks + Transitions
for i, dense_block in enumerate(densenet.dense_blocks):
    X = dense_block(X)
    print(f"Dense Block {i+1}: {list(X.shape)}")

    if i < len(densenet.transitions):
        X = densenet.transitions[i](X)
        print(f"  → Transition: {list(X.shape)}")

# 分类器
X = densenet.classifier(X)
print(f"\n最终输出: {list(X.shape)}")

DenseNet各阶段输出形状:
------------------------------------------------------------
初始Conv: [1, 24, 56, 56]
Dense Block 1: [1, 72, 56, 56]
  → Transition: [1, 36, 28, 28]
Dense Block 2: [1, 84, 28, 28]
  → Transition: [1, 42, 14, 14]
Dense Block 3: [1, 90, 14, 14]
  → Transition: [1, 45, 7, 7]
Dense Block 4: [1, 93, 7, 7]

最终输出: [1, 10]


### 3.6 DenseNet变体

| 模型 | 配置 | 参数量 | Top-1错误率 |
|------|------|--------|-------------|
| DenseNet-121 | (6,12,24,16), k=32 | 8M | 25.0% |
| DenseNet-169 | (6,12,32,32), k=32 | 14M | 24.0% |
| DenseNet-201 | (6,12,48,32), k=32 | 20M | 23.0% |
| DenseNet-264 | (6,12,64,48), k=32 | 33M | 22.1% |

**对比ResNet**:
- DenseNet-201 (20M) vs ResNet-101 (45M)
- 参数少56%,性能相当!

---

## 第四部分: 技术对比与总结

### 4.1 三大技术对比

In [9]:
import pandas as pd

data = {
    '技术': ['BatchNorm', 'NiN', 'DenseNet'],
    '年份': [2015, 2013, 2017],
    '核心创新': ['归一化+缩放', '1×1卷积+GAP', '密集连接'],
    '主要作用': ['训练加速', '参数减少', '特征复用'],
    '应用场景': ['几乎所有模型', '降维/增加非线性', '中小型模型'],
    '计算开销': ['中等', '低', '高(内存)'],
}

df = pd.DataFrame(data)
print("\n现代CNN关键技术对比:")
print("=" * 80)
print(df.to_string(index=False))
print("=" * 80)


现代CNN关键技术对比:
       技术   年份      核心创新 主要作用     应用场景  计算开销
BatchNorm 2015    归一化+缩放 训练加速   几乎所有模型    中等
      NiN 2013 1×1卷积+GAP 参数减少 降维/增加非线性     低
 DenseNet 2017      密集连接 特征复用    中小型模型 高(内存)


### 4.2 技术组合

**现代架构通常组合使用**:

1. **ResNet + BatchNorm**
   ```python
   Conv → BN → ReLU → Conv → BN → (+) → ReLU
   ```

2. **Inception + 1×1卷积**
   ```python
   并行: [1×1, 1×1→3×3, 1×1→5×5, pool→1×1]
   ```

3. **DenseNet + BatchNorm + 1×1卷积**
   ```python
   Dense Block: BN → ReLU → Conv
   Transition: BN → ReLU → Conv1×1 → AvgPool
   ```

### 4.3 设计原则

**从这些技术学到的**:

1. **归一化很重要** (BatchNorm)
   - 几乎所有模型都用
   - 加速训练,提高稳定性

2. **1×1卷积是瑞士军刀** (NiN)
   - 降维/升维
   - 增加非线性
   - 跨通道交互

3. **跳跃连接改变游戏** (ResNet, DenseNet)
   - 梯度流动
   - 特征复用
   - 更深的网络

4. **全局平均池化代替全连接** (NiN)
   - 减少参数
   - 更好泛化
   - 位置不变性

5. **模块化设计** (所有)
   - 重复使用块
   - 易于理解和修改
   - 便于实验

---

## 小结

### Batch Normalization
- **问题**: 内部协变量偏移,训练不稳定
- **方案**: 归一化 + 可学习缩放/偏移
- **效果**: 加速收敛,允许更大学习率,正则化
- **使用**: Conv/FC → **BN** → Activation

### Network in Network
- **问题**: 全连接层参数爆炸
- **方案**: 1×1卷积 + 全局平均池化
- **效果**: 大幅减少参数,更好泛化
- **影响**: 现代架构的标准组件

### DenseNet
- **问题**: 特征复用不充分
- **方案**: 密集连接 (每层连所有前层)
- **效果**: 参数效率高,梯度流动好
- **代价**: 内存消耗大

### 现代CNN的"标准配方"

```python
1. 归一化: BatchNorm / LayerNorm
2. 激活: ReLU / GELU
3. 跳跃连接: ResNet-style / DenseNet-style
4. 1×1卷积: 降维/升维/增加非线性
5. 全局池化: 代替全连接
6. 模块化: 重复使用块
```

## 练习

1. **BatchNorm实验**: 对比有/无BatchNorm的训练速度
2. **1×1卷积**: 计算使用1×1降维节省的计算量
3. **DenseNet分析**: 计算Dense Block的内存消耗
4. **技术组合**: 设计自己的CNN,组合这些技术
5. **参数对比**: 对比AlexNet, NiN, DenseNet的参数量
6. **可视化**: 可视化DenseNet的密集连接模式